# Netflix Content Analysis — EDA

In [1]:
import pandas as pd
import numpy as np

df         = pd.read_csv('../data/processed/netflix_clean.csv', parse_dates=['date_added'])
df_genres  = pd.read_csv('../data/processed/netflix_genres.csv')
df_country = pd.read_csv('../data/processed/netflix_country.csv')

print(f'Main: {len(df):,} rows | Genres: {len(df_genres):,} | Country: {len(df_country):,}')

Main: 8,797 rows | Genres: 19,303 | Country: 10,835


In [2]:
# movies vs tv shows
type_counts = df['type'].value_counts()
for t, n in type_counts.items():
    print(f'{t}: {n:,} ({n/len(df)*100:.1f}%)')

Movie: 6,131 (69.7%)
TV Show: 2,666 (30.3%)


In [3]:
print('Top genres overall:')
print(df_genres['listed_in'].value_counts().head(15))

print('\nMovies:')
print(df_genres[df_genres['type']=='Movie']['listed_in'].value_counts().head(10))

print('\nTV Shows:')
print(df_genres[df_genres['type']=='TV Show']['listed_in'].value_counts().head(10))

Top genres overall:
listed_in
International Movies        2752
Dramas                      2427
Comedies                    1674
International TV Shows      1350
Documentaries                869
Action & Adventure           859
TV Dramas                    762
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Thrillers                    577
TV Comedies                  574
Crime TV Shows               469
Kids' TV                     449
Docuseries                   394
Name: count, dtype: int64

Movies:
listed_in
International Movies        2752
Dramas                      2427
Comedies                    1674
Documentaries                869
Action & Adventure           859
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Thrillers                    577
Music & Musicals             375
Name: count, dtype: int64

TV Shows:
listed_in
International TV Shows    1350
TV Dramas             

In [4]:
# content added per year
yearly = df.groupby('year_added')['show_id'].count()
print(yearly.to_string())
print(f'\nPeak: {yearly.idxmax()} ({yearly.max():,} titles)')

year_added
2008       2
2009       2
2010       1
2011      13
2012       3
2013      11
2014      24
2015      82
2016     429
2017    1188
2018    1649
2019    2016
2020    1879
2021    1498

Peak: 2019 (2,016 titles)


In [5]:
df.groupby(['year_added', 'type'])['show_id'].count().unstack().fillna(0)

type,Movie,TV Show
year_added,,
2008,1.0,1.0
2009,2.0,0.0
2010,1.0,0.0
2011,13.0,0.0
2012,3.0,0.0
2013,6.0,5.0
2014,19.0,5.0
2015,56.0,26.0
2016,253.0,176.0


In [6]:
top_countries = df_country['country'].value_counts().head(15)
for c, n in top_countries.items():
    print(f'{c:<30} {n:>5}')

United States                   4513
India                           1046
United Kingdom                   803
Canada                           445
France                           393
Japan                            317
Spain                            232
South Korea                      231
Germany                          226
Mexico                           169
China                            162
Australia                        159
Egypt                            117
Turkey                           113
Hong Kong                        105


In [7]:
print(df['rating'].value_counts())
print()
df.groupby(['rating', 'type'])['show_id'].count().unstack().fillna(0)

rating
TV-MA       3209
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
74 min         1
84 min         1
66 min         1
Name: count, dtype: int64



type,Movie,TV Show
rating,,
66 min,1.0,0.0
74 min,1.0,0.0
84 min,1.0,0.0
G,41.0,0.0
NC-17,3.0,0.0
NR,75.0,4.0
PG,287.0,0.0
PG-13,490.0,0.0
R,797.0,2.0


In [8]:
movies = df[df['type'] == 'Movie']
shows  = df[df['type'] == 'TV Show']

print('Movie duration (mins):')
print(movies['duration_mins'].describe().round(1))

print('\nTV Show seasons:')
print(shows['duration_seasons'].describe().round(1))

print(f'\n{(shows["duration_seasons"]==1).sum()/len(shows)*100:.1f}% of shows have only 1 season')
print(f'{(movies["duration_mins"]>120).sum()/len(movies)*100:.1f}% of movies are over 2 hours')

Movie duration (mins):
count    6128.0
mean       99.6
std        28.3
min         3.0
25%        87.0
50%        98.0
75%       114.0
max       312.0
Name: duration_mins, dtype: float64

TV Show seasons:
count    2666.0
mean        1.8
std         1.6
min         1.0
25%         1.0
50%         1.0
75%         2.0
max        17.0
Name: duration_seasons, dtype: float64

67.3% of shows have only 1 season
18.6% of movies are over 2 hours


In [9]:
print(df[df['director'] != 'Unknown']['director'].value_counts().head(10))

director
Rajiv Chilaka             19
Raúl Campos, Jan Suter    18
Marcus Raboy              16
Suhas Kadav               16
Jay Karas                 14
Cathy Garcia-Molina       13
Martin Scorsese           12
Youssef Chahine           12
Jay Chapman               12
Steven Spielberg          11
Name: count, dtype: int64


In [10]:
df_cast = (df[df['cast'] != 'Unknown']
           .assign(cast=lambda x: x['cast'].str.split(', '))
           .explode('cast')
           .assign(cast=lambda x: x['cast'].str.strip()))

print(df_cast['cast'].value_counts().head(15))

cast
Anupam Kher         43
Shah Rukh Khan      35
Julie Tejwani       33
Takahiro Sakurai    32
Naseeruddin Shah    32
Rupa Bhimani        31
Om Puri             30
Akshay Kumar        30
Yuki Kaji           29
Paresh Rawal        28
Amitabh Bachchan    28
Boman Irani         27
Rajesh Kava         26
Vincent Tong        26
Andrea Libman       25
Name: count, dtype: int64


In [11]:
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('month_name')['show_id'].count().reindex(month_order)
print(monthly.to_string())
print(f'\nBusiest: {monthly.idxmax()} | Quietest: {monthly.idxmin()}')

month_name
Jan    738
Feb    563
Mar    742
Apr    764
May    632
Jun    728
Jul    827
Aug    755
Sep    770
Oct    760
Nov    705
Dec    813

Busiest: Jul | Quietest: Feb


---
## Key Findings

- Content mix: **69.7% movies**, **30.3% TV shows**
- Top genre: **International Movies** with 2,752 titles
- Content peaked in **2019**
- **United States** leads, India ranks #2 with 1,046 titles
- Most common rating: **TV-MA** (3,209 titles)
- Average movie: **~99 minutes**
- 67.3% of shows have only 1 season
- Most content added in **July**